# RaspTank initial validation
Run discovery and symbolic cells on any computer. Imports alone never open devices.
These checks validate software, not physical wiring, direction or calibration.
Only the final cell can access hardware, and it is disabled by default.


In [ ]:
from pathlib import Path
import sys
repo_root = next((p for p in (Path.cwd(), Path.cwd().parent)
                  if (p / 'rasptank' / '__init__.py').is_file()), None)
if repo_root is None:
    raise RuntimeError('Open this notebook from the repository or notebooks directory')
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
# File discovery does not import parent packages as dotted find_spec can.
for name in ('config.py', 'exceptions.py', 'tank.py', 'hardware/motors.py',
             'hardware/servos.py', 'hardware/ultrasonic.py', 'hardware/leds.py',
             'hardware/display.py'):
    print(name, (repo_root / 'rasptank' / name).is_file())


In [ ]:
import rasptank
from rasptank import RaspTank, MotorController, ServoController, UltrasonicSensor
from rasptank.hardware.leds import LEDStrip
from rasptank.hardware.display import OledDisplay
from rasptank.config import SERVO_RANGES, SERVO_DIRECTIONS
from rasptank.exceptions import HardwareDisabledError
print('IMPORT_OK', rasptank.__all__)


In [ ]:
tank = RaspTank(use_gpio=False, enable_servos=True, enable_ultrasonic=True,
                enable_leds=True, enable_display=True)
try:
    tank.forward(15)
    tank.backward(15)
    tank.turn_left(15)
    tank.turn_right(15)
    tank.stop()
    assert tank.motors.pwm_a is None
    for channel, limits in enumerate(SERVO_RANGES):
        print('servo', channel, 'range', limits, 'direction', SERVO_DIRECTIONS[channel])
    tank.servos.set_pulse(0, 300)
    assert tank.servos.move_relative(0, 10) == 310
    assert (tank.ultrasonic.trigger_pin, tank.ultrasonic.echo_pin) == (11, 8)
    try:
        tank.ultrasonic.distance()
    except HardwareDisabledError:
        print('No fabricated dry-run distance')
    tank.leds.color_wipe(10, 20, 30)
    tank.display.show('SYMBOLIC', 'NO HARDWARE')
    print('SYMBOLIC_OK')
finally:
    tank.close()


## Optional manual Pi motor check
Stop the legacy server first; never share GPIO/PCA9685 ownership. Verify the Pi kernel,
BCM wiring and power, secure the robot on a stand with tracks clear, and keep a physical
power cutoff ready. A table alone is not a restraint. Read README manual steps first.
No servo motion is included. Initial 300 values are assumptions, not measured positions;
new API calibration is transient and does not read or rewrite legacy calibration files.
A successful symbolic test does not establish physical safety or hardware correctness.


In [ ]:
RUN_HARDWARE_TEST = False
if RUN_HARDWARE_TEST:
    import time
    hardware_tank = None
    try:
        hardware_tank = RaspTank(use_gpio=True)
        hardware_tank.forward(15)
        time.sleep(0.3)
        hardware_tank.stop()
    finally:
        if hardware_tank is not None:
            hardware_tank.close()
else:
    print('HARDWARE_TEST_DISABLED')
